In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.obs.lsst.translators.lsst import SIMONYI_LOCATION
from lsst.geom import SpherePoint,Angle,Extent2I,Box2I,Extent2D,Point2D, Point2I
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5, SkyCoord, angular_separation
import astropy.units as u
from astropy.time import Time, TimeDelta
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.daf.butler import Butler
from lsst.obs.lsst import LsstCam
from lsst.obs.lsst.cameraTransforms import LsstCameraTransforms
import lsst.geom as geom



In [ ]:
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=False)
instrument = 'LSSTCam'


In [ ]:
from datetime import datetime

# WGS84 constants
R_EARTH = 6378.137       # km
F_EARTH = 1 / 298.257223563
MU_EARTH = 398600.4418  # km^3 / s^2
E2 = F_EARTH * (2 - F_EARTH)

R_GEO = 42164.0          # km


In [ ]:
def geodetic_to_ecef(lat_deg, lon_deg, alt_km):
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    sin_lat = np.sin(lat)
    cos_lat = np.cos(lat)

    N = R_EARTH / np.sqrt(1 - E2 * sin_lat**2)

    x = (N + alt_km) * cos_lat * np.cos(lon)
    y = (N + alt_km) * cos_lat * np.sin(lon)
    z = (N * (1 - E2) + alt_km) * sin_lat

    return np.array([x, y, z])


In [ ]:
def azel_to_enu(az_deg, el_deg):
    az = np.deg2rad(az_deg)
    el = np.deg2rad(el_deg)

    e = np.cos(el) * np.sin(az)
    n = np.cos(el) * np.cos(az)
    u = np.sin(el)

    return np.array([e, n, u])


In [ ]:
def enu_to_ecef_matrix(lat_deg, lon_deg):
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    sin_lat = np.sin(lat)
    cos_lat = np.cos(lat)
    sin_lon = np.sin(lon)
    cos_lon = np.cos(lon)

    return np.array([
        [-sin_lon,              -sin_lat * cos_lon,   cos_lat * cos_lon],
        [ cos_lon,              -sin_lat * sin_lon,   cos_lat * sin_lon],
        [ 0.0,                   cos_lat,             sin_lat          ]
    ])


In [ ]:
def utc_to_gmst(time_utc):
    """
    Compute Greenwich Mean Sidereal Time (radians).
    time_utc: datetime (UTC)
    """
    if isinstance(time_utc, str):
        time_utc = datetime.fromisoformat(time_utc.replace("Z", ""))

    jd = (
        time_utc.timestamp() / 86400.0
        + 2440587.5
    )
    T = (jd - 2451545.0) / 36525.0
    gmst = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - T**3 / 38710000.0
    )

    return np.deg2rad(gmst % 360.0)


In [ ]:
def ecef_to_eci(r_ecef, time_utc):
    """Rotate ECEF vector into ECI using GMST."""
    theta = utc_to_gmst(time_utc)

    c = np.cos(theta)
    s = np.sin(theta)

    R = np.array([
        [ c, -s, 0.0],
        [ s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

    return R @ r_ecef

In [ ]:
def ray_consistent_with_geo(
    az_deg,
    el_deg,
    time_utc,
    lat_deg,
    lon_deg,
    alt_km,
    imax_deg=0.2,     # max GEO inclination
    r_tol_km=200.0    # GEO radius tolerance
):
    """
    Full GEO consistency test:
    - GEO radius shell
    - Equatorial-band constraint (|lat| < imax)

    Returns
    -------
    is_geo : bool
    intersections : list of dicts with keys:
        s_km, r_ecef, r_eci, geocentric_lat_deg
    """

    # Observer position
    r_obs = geodetic_to_ecef(lat_deg, lon_deg, alt_km)  
    # LOS in ECEF
    rho_enu = azel_to_enu(az_deg, el_deg)
    R = enu_to_ecef_matrix(lat_deg, lon_deg)
    rho_hat = R @ rho_enu
    rho_hat /= np.linalg.norm(rho_hat)

    b = 2.0 * np.dot(r_obs, rho_hat)
    c0 = np.dot(r_obs, r_obs)

    intersections = []

    for Rg in (R_GEO - r_tol_km, R_GEO + r_tol_km):
        c = c0 - Rg**2
        disc = b*b - 4.0*c
        if disc < 0:
            continue

        sqrt_disc = np.sqrt(disc)
        for s in [(-b + sqrt_disc)/2.0, (-b - sqrt_disc)/2.0]:
            if s <= 0:
                continue

            r_ecef = r_obs + s * rho_hat
            r_eci = ecef_to_eci(r_ecef, time_utc)

            rmag = np.linalg.norm(r_eci)
            lat = np.arcsin(r_eci[2] / rmag)
            lat_deg = np.rad2deg(lat)
            if abs(lat_deg) <= imax_deg:
                intersections.append({
                    "s_km": s,
                    "r_ecef": r_ecef,
                    "r_eci": r_eci,
                    "geocentric_lat_deg": lat_deg
                })

    return (len(intersections) > 0), intersections            
            

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)


def angular_rate(rho_hats, times):
    """
    Estimate angular rate magnitude (rad/s) from multiple unit vectors.
    """
    omegas = []

    for i in range(len(rho_hats) - 1):
        dt = (times[i+1] - times[i])
        cosang = np.clip(np.dot(rho_hats[i], rho_hats[i+1]), -1, 1)
        dtheta = np.arccos(cosang)
        omegas.append(dtheta / dt)
        print(dtheta / dt)

    return np.median(omegas)

In [ ]:
lat_deg=SIMONYI_LOCATION.lat.deg
lon_deg=SIMONYI_LOCATION.lon.deg

rho_hats = []
for az, alt in zip(azs, alts):
    print(az, alt)
    rho_enu = azel_to_enu(az, alt)
    print(rho_enu)
    R = enu_to_ecef_matrix(lat_deg, lon_deg)
    rho_hat = R @ rho_enu
    rho_hat /= np.linalg.norm(rho_hat)
    print(rho_hat)
    rho_hats.append(rho_hat)

In [ ]:
def is_consistent_with_leo(
    rho_hats,          # list of LOS unit vectors (ECI)
    times,             # list of datetime UTC
    h_min=300.0,       # km
    h_max=2000.0,      # km
    v_tol=1.5          # km/s tolerance
):
    """
    Test whether angular motion is consistent with a LEO object.
    """

    omega = angular_rate(rho_hats, times)  # rad/s
    print(omega)
    for h in np.linspace(h_min, h_max, 50):
        r = R_EARTH + h
        v_trans = omega * r

        v_circ = np.sqrt(MU_EARTH / r)

        if abs(v_trans - v_circ) < v_tol:
            return True, {
                "alt_km": h,
                "omega_rad_s": omega,
                "v_trans_km_s": v_trans,
                "v_circ_km_s": v_circ
            }

    return False, None    

In [ ]:
rho_hats

In [ ]:
is_consistent_with_leo(
    rho_hats,          # list of LOS unit vectors (ECI)
    time_secs,             # list of datetime UTC
    h_min=300.0,       # km
    h_max=50000.0,      # km
    v_tol=1.0         # km/s tolerance
)

In [ ]:
#expId = 2025052300078
#RA = 149.614382
#Dec = 0.483682
#time = Time("2025-05-23T23:08:28.62", scale='tai')
#expId = 2025052300072
#RA = 150.925074
#Dec = 2.812925
#time = Time("2025-05-23T23:02:42.33", scale='tai')
expId = 2025052800718
RA = 308.601079
Dec = -36.964745
time = Time("2025-05-29T09:33:27.57", scale='tai')
skyLocation = SkyCoord(RA*u.deg, Dec*u.deg)
altAz = AltAz(obstime=time, location=SIMONYI_LOCATION)
obsAltAz = skyLocation.transform_to(altAz)
print(obsAltAz.alt.degree, obsAltAz.az.degree)

In [ ]:
is_geo, hits = ray_consistent_with_geo(
    az_deg=obsAltAz.az.degree,
    el_deg=obsAltAz.alt.degree,
    time_utc="2025-05-23T23:08:28.62",
    lat_deg=SIMONYI_LOCATION.lat.deg,
    lon_deg=SIMONYI_LOCATION.lon.deg,
    alt_km=SIMONYI_LOCATION.height.value / 1000.0,
    imax_deg=0.50
)

print("Consistent with GEO?", is_geo)
for h in hits:
    print(h["geocentric_lat_deg"], "deg")

In [ ]:
dayObs = 20251114
seqs = [286, 294, 297, 302, 304, 309]
RAs = [14.093308, 15.494940, 15.714306, 16.197187, 17.808199, 19.359924]
Decs = [-44.609046,  -45.033772,  -42.905744, -43.824294, -44.559334, -44.937913]     
times = ['2025-11-15T04:49:27.56', '2025-11-15T05:02:59.56', '2025-11-15T05:05:26.27',
         '2025-11-15T05:10:09.83', '2025-11-15T05:11:58.50', '2025-11-15T05:16:10.94']
detectors = [83, 62, 187, 161, 103, 60]

In [ ]:
dayObs = 20250723
seqs = [23, 27]
RAs = [225.7947, 225.9280]
Decs = [-38.1330, -37.9893]     
times = ['2025-07-23T23:51:18.48', '2025-07-23T23:54:00.58']

In [ ]:
dayObs = 20250523
seqs = [72, 78]
RAs = [150.9251, 149.6144]
Decs = [2.8129, 0.4837]     
times = ['2025-05-23T23:02:42.33', '2025-05-23T23:08:28.62']
detectors = [151, 33]

In [ ]:
wavelengths = {'u':3671, 'g':4827, 'r':6223, 'i':7546, 'z':8691, 'y':9712}
camera = LsstCam.getCamera()

alts = []
azs = []
time_secs = []
plot_ras = []
plot_decs = []
for i in range(6):
    #time = Time(times[i], scale='tai') + TimeDelta(30.0, format='sec')
    skyLocation = SkyCoord(RAs[i]*u.deg, Decs[i]*u.deg)
    expId = int(dayObs * 1E5 + seqs[i])
    rawExp = butler.get('raw', detector=detectors[i], exposure=expId, instrument=instrument)
    calExp = butler.get('preliminary_visit_image', detector=detectors[i], visit=expId, instrument=instrument)
    md = rawExp.getMetadata()
    filter = md['FILTBAND']
    wl = wavelengths[filter] * u.angstrom
    pressure = md['PRESSURE'] * u.pascal
    temp = md['AIRTEMP'] * u.deg_C
    hum = md['HUMIDITY'] / 100.0
    time = Time((md['MJD-BEG'] + md['MJD-END']) / 2.0, format='mjd', scale='tai')

    altAz = AltAz(obstime=time, location=SIMONYI_LOCATION, pressure=pressure, 
                 temperature=temp, relative_humidity=hum, obswl=wl)
    obsAltAz = skyLocation.transform_to(altAz)
    alts.append(obsAltAz.alt.degree)
    azs.append(obsAltAz.az.degree)
    time_secs.append(time.unix_tai)
    is_geo, hits = ray_consistent_with_geo(
        az_deg=obsAltAz.az.degree,
        el_deg=obsAltAz.alt.degree,
        time_utc=times[i],
        lat_deg=SIMONYI_LOCATION.lat.deg,
        lon_deg=SIMONYI_LOCATION.lon.deg,
        alt_km=SIMONYI_LOCATION.height.value / 1000.0,
        imax_deg=4.0,     # max GEO inclination
        r_tol_km=1000.0    # GEO radius tolerance
    )
    plot_ras.append(RAs[i])
    plot_decs.append(Decs[i])
    print(seqs[i], is_geo, obsAltAz.alt.degree, md['ELSTART'], obsAltAz.az.degree, md['AZSTART'])
    wcs = rawExp.getWcs()
    """
    if wcs:
        print("Using calExp")
    else:
        wcs = rawExp.getWcs()
        print("Using rawExp")
    """
    sp = SpherePoint(RAs[i]*geom.degrees, Decs[i]*geom.degrees)
    pixel = wcs.skyToPixel(sp)
    #print(seqs[i], pixel)
    detector = camera[detectors[i]]
    detName = detector.getName()
    bbox = detector.getBBox()
    nx,ny = bbox.getDimensions()
    lct = LsstCameraTransforms(camera,detName)
    llfpX, llfpY = lct.ccdPixelToFocalMm(0, 0, detName)
    urfpX, urfpY = lct.ccdPixelToFocalMm(nx, ny, detName)
    llAz = (md['AZSTART'] + md['AZEND']) / 2.0 + llfpY * 1000 / 10 * 0.2 / 3600.0
    urAz = (md['AZSTART'] + md['AZEND']) / 2.0 + urfpY * 1000 / 10 * 0.2 / 3600.0  
    print(llAz, obsAltAz.az.degree-360.0, urAz)
        

In [ ]:
lat_deg=SIMONYI_LOCATION.lat.deg
lon_deg=SIMONYI_LOCATION.lon.deg

rho_hats = []
for az, alt in zip(azs, alts):
    print(az, alt)
    rho_enu = azel_to_enu(az, alt)
    print(rho_enu)
    R = enu_to_ecef_matrix(lat_deg, lon_deg)
    rho_hat = R @ rho_enu
    rho_hat /= np.linalg.norm(rho_hat)
    print(rho_hat)
    rho_hats.append(rho_hat)

In [ ]:
wcs

In [ ]:
time_secs = np.array(time_secs)
time_secs -= time_secs[0] - 1.0


In [ ]:
time_secs

In [ ]:
is_consistent_with_leo(
    rho_hats,          # list of LOS unit vectors (ECI)
    time_secs,             # list of datetime UTC
    h_min=5000.0,       # km
    h_max=50000.0,      # km
    v_tol=1.0         # km/s tolerance
)

In [ ]:
i0 = 5
i1 = 4

dRA = RAs[i1] - RAs[i0]
dDec = Decs[i1] - Decs[i0]
deg_per_sec = (np.sqrt(dDec**2 + (dRA * np.cos(Decs[0] * np.pi / 180.0))**2)) / (time_secs[i1] - time_secs[i0])
rad_per_sec = deg_per_sec / (180.0 / np.pi)
rad_per_sec

In [ ]:
#2026010500024 - GEOs
track = 746 # pixels
length = track * 0.2 / 3600.0 / (180.0 / np.pi) # radians
print(length)
rad_per_sec = length / 10.0
rad_per_sec

In [ ]:
360 / (180.0 / np.pi) / 86400

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(15,5))
plt.subplots_adjust(wspace=0.5)
axs[0].set_ylabel('Azimuth')
axs[0].set_xlabel('Time(sec)')
axs[0].scatter(time_secs, azs, marker='x')
axs[1].set_ylabel('Altitude')
axs[1].set_xlabel('Time(sec)')
axs[1].scatter(time_secs, alts, marker='x')
axs[2].set_ylabel('Azimuth')
axs[2].set_xlabel('Altitude')
axs[2].scatter(azs, alts, marker='x', c=time_secs, cmap='coolwarm')
axs[2].plot(azs, alts)
plt.savefig(f"/home/c/cslage/u/LSSTCam/images/Transients_vs_AltAz_07Feb26.png")

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(15,5))
plt.subplots_adjust(wspace=0.5)
axs[0].set_ylabel('RA')
axs[0].set_xlabel('Time(sec)')
axs[0].scatter(time_secs, plot_ras, marker='x')
axs[1].set_ylabel('Dec')
axs[1].set_xlabel('Time(sec)')
axs[1].scatter(time_secs, plot_decs, marker='x')
axs[2].set_xlabel('RA')
axs[2].set_ylabel('Dec')
axs[2].scatter(plot_ras, plot_decs, marker='x', c=time_secs, cmap='coolwarm')
axs[2].plot(plot_ras, plot_decs)
plt.savefig(f"/home/c/cslage/u/LSSTCam/images/Transients_vs_RADec_07Feb26.png")